In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install chromadb
!pip install pypdf
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 일본_탈탄소 성장형 경제구조로의 원활한 이행을 위한 저탄소 수소 등의 공급 및 이용 촉진에 관한 법률_원문본(2024.05.24.공포, 2024.10.23.시행) (1).pdf to 일본_탈탄소 성장형 경제구조로의 원활한 이행을 위한 저탄소 수소 등의 공급 및 이용 촉진에 관한 법률_원문본(2024.05.24.공포, 2024.10.23.시행) (1).pdf


In [ ]:
import os

pdf_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

print("업로드된 PDF:", pdf_files)

업로드된 PDF: ['일본_탈탄소 성장형 경제구조로의 원활한 이행을 위한 저탄소 수소 등의 공급 및 이용 촉진에 관한 법률_원문본(2024.05.24.공포, 2024.10.23.시행) (1).pdf']


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    docs = loader.load()

    for doc in docs:
        doc.metadata["country"] = "Japan"
        doc.metadata["language"] = "Japanese"
        doc.metadata["file_name"] = pdf_file

    documents.extend(docs)

print("불러온 전체 페이지 수:", len(documents))

/tmp/ipykernel_898/3767544796.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


불러온 전체 페이지 수: 28


In [ ]:
from langchain_core.documents import Document

full_text = "\n\n".join(
    doc.page_content.strip()
    for doc in documents
    if doc.page_content.strip()
)

combined_document = Document(
    page_content=full_text,
    metadata={
        "country": "Japan",
        "language": "Japanese",
        "law_name": "脱炭素成長型経済構造への円滑な移行のための低炭素水素等の供給及び利用の促進に関する法律",
        "file_name": pdf_files[0],
        "total_pages": len(documents)
    }
)

print("통합 문서 글자 수:", len(combined_document.page_content))

통합 문서 글자 수: 26054


In [ ]:
import re

clean_text = combined_document.page_content

clean_text = re.sub(
    r"\n?\s*-\s*\d+\s*-\s*\n?",
    "\n",
    clean_text
)

clean_text = re.sub(r"\n{3,}", "\n\n", clean_text).strip()

body_start_match = re.search(
    r"（目的）\s*\n?\s*第一条",
    clean_text
)

if body_start_match is None:
    raise ValueError("본문의 （目的） 第一条 시작점을 찾지 못했습니다.")

preamble_text = clean_text[:body_start_match.start()].strip()
body_text = clean_text[body_start_match.start():].strip()

print("정리 후 전체 글자 수:", len(clean_text))
print("목차·머리말 글자 수:", len(preamble_text))
print("법령 본문 글자 수:", len(body_text))
print(body_text[:200])

정리 후 전체 글자 수: 25811
목차·머리말 글자 수: 347
법령 본문 글자 수: 25462
（目的） 
第一条 この法律は、世界的規模でエネルギーの脱炭素化に向けた取組等が進めら
れる中で、我が国における低炭素水素等の供給及び利用を早期に促進するため、
低炭素水素等の供給及び利用の促進に関する基本方針の策定、低炭素水素等供給
等事業に関する計画の認定等の措置を講ずることにより、エネルギーの安定的か
つ低廉な供給を確保しつつ、脱炭素成長型経済構造（脱炭素成長型経済構造への
円滑な移行の推進


In [ ]:
import re

# 1. 실제 조문 시작점 찾기
article_header_pattern = re.compile(
    r"""
    ^[ \t]*
    (?:
        （(?P<title>[^）\n]+)）[ \t]*\n
        [ \t]*
    )?
    (?P<article>
        第[一二三四五六七八九十百〇零]+条
        (?:の[一二三四五六七八九十百〇零]+)?
    )
    (?=[ \t])
    """,
    re.MULTILINE | re.VERBOSE
)

matches = list(article_header_pattern.finditer(body_text))
print("찾은 실제 조문 시작점 수:", len(matches))


# 2. 조문 블록 생성
article_blocks_raw = []

for i, match in enumerate(matches):
    start = match.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(body_text)

    article_blocks_raw.append({
        "article": match.group("article"),
        "title": match.group("title") or "",
        "text": body_text[start:end].strip()
    })

print("필터링 전 조문 블록 수:", len(article_blocks_raw))


# 3. 이 법률의 정상 조문 순서
main_articles = [
    "第一条", "第二条", "第三条", "第四条", "第五条", "第六条",
    "第七条", "第八条", "第九条", "第十条", "第十一条", "第十二条",
    "第十三条", "第十四条", "第十五条", "第十六条", "第十七条",
    "第十八条", "第十九条", "第二十条", "第二十一条", "第二十二条",
    "第二十三条", "第二十四条", "第二十五条", "第二十六条",
    "第二十七条", "第二十八条", "第二十九条", "第三十条",
    "第三十一条", "第三十二条", "第三十三条", "第三十四条",
    "第三十五条", "第三十六条", "第三十七条", "第三十八条",
    "第三十九条", "第四十条", "第四十一条", "第四十二条",
    "第四十三条", "第四十四条", "第四十五条", "第四十六条",
    "第四十七条", "第四十八条", "第四十九条", "第五十条",
    "第五十一条", "第五十二条"
]

supplementary_articles = [
    "第一条", "第二条", "第三条", "第四条", "第十四条"
]

expected_articles = main_articles + supplementary_articles


# 4. 정상 순서대로 필요한 조문만 필터링
filtered_blocks = []
search_start = 0

for expected_article in expected_articles:
    found = False

    for i in range(search_start, len(article_blocks_raw)):
        if article_blocks_raw[i]["article"] == expected_article:
            filtered_blocks.append(article_blocks_raw[i])
            search_start = i + 1
            found = True
            break

    if not found:
        print("⚠️ 찾지 못한 조문:", expected_article)

article_blocks = filtered_blocks

print("필터링 후 조문 블록 수:", len(article_blocks))

찾은 실제 조문 시작점 수: 62
필터링 전 조문 블록 수: 62
필터링 후 조문 블록 수: 57


In [ ]:
for i, block in enumerate(article_blocks):
    print(i, block["article"], block["title"])


0 第一条 目的
1 第二条 定義
2 第三条 基本方針
3 第四条 国の責務
4 第五条 関係地方公共団体の責務
5 第六条 事業者の責務
6 第七条 計画の認定
7 第八条 計画の変更等
8 第九条 地位の承継
9 第十条 
10 第十一条 
11 第十二条 製造の承認
12 第十三条 製造の承認の地位の承継
13 第十四条 製造の変更の承認
14 第十五条 製造の開始等の届出
15 第十六条 承認製造者等に関する高圧ガス保安法の準用
16 第十七条 貯蔵所の承認
17 第十八条 貯蔵所の承認の地位の承継
18 第十九条 貯蔵所の変更の承認
19 第二十条 貯蔵の開始等の届出
20 第二十一条 承認貯蔵所の所有者又は占有者等に関する高圧ガス保安法の準用
21 第二十二条 輸入検査の認定等
22 第二十三条 承認の取消し等
23 第二十四条 通知等
24 第二十五条 高圧ガス保安法の特例
25 第二十六条 完成検査等に関する高圧ガス保安法の適用
26 第二十七条 高圧ガス保安協会の業務等
27 第二十八条 聴聞の特例
28 第二十九条 審査請求の手続における意見の聴取
29 第三十条 審査請求の制限
30 第三十一条 
31 第三十二条 水素等供給事業者の判断の基準となるべき事項
32 第三十三条 指導及び助言
33 第三十四条 勧告及び命令
34 第三十五条 資金の確保
35 第三十六条 承認の条件
36 第三十七条 報告の徴収
37 第三十八条 立入検査
38 第三十九条 手数料
39 第四十条 大都市の特例
40 第四十一条 協議
41 第四十二条 主務大臣等
42 第四十三条 環境大臣との関係
43 第四十四条 権限の委任
44 第四十五条 省令への委任
45 第四十六条 経過措置
46 第四十七条 
47 第四十八条 
48 第四十九条 
49 第五十条 
50 第五十一条 
51 第五十二条 
52 第一条 施行期日
53 第二条 検討
54 第三条 調整規定
55 第四条 
56 第十四条 政令への委任


In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

sub_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1400,
    chunk_overlap=250,
    separators=[
        "\n１０ ", "\n９ ", "\n８ ", "\n７ ", "\n６ ",
        "\n５ ", "\n４ ", "\n３ ", "\n２ ",
        "\n一 ", "\n二 ", "\n三 ", "\n四 ", "\n五 ",
        "\n六 ", "\n七 ", "\n八 ", "\n九 ", "\n十 ",
        "\n\n", "\n", "。"
    ]
)

chunks = []

for block in article_blocks:
    article_number = block["article"]
    article_title = block["title"]
    block_text = block["text"]

    metadata = {
        **combined_document.metadata,
        "document_section": "법령 본문",
        "article": article_number,
        "article_title": article_title
    }

    if len(block_text) <= 1400:
        chunks.append(
            Document(
                page_content=block_text,
                metadata=metadata
            )
        )

    else:
        parts = sub_splitter.split_text(block_text)

        header = article_number
        if article_title:
            header = f"（{article_title}）\n{article_number}"

        for part_index, part in enumerate(parts):
            if part_index > 0:
                part = (
                    f"{header}\n"
                    f"[조문 계속 {part_index + 1}]\n"
                    f"{part}"
                )

            chunks.append(
                Document(
                    page_content=part,
                    metadata={
                        **metadata,
                        "part": part_index + 1
                    }
                )
            )

print("최종 청크 수:", len(chunks))

최종 청크 수: 68


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

/tmp/ipykernel_898/1224575925.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [ ]:
# 마지막 5개는 부칙 조문
for chunk in chunks[-5:]:
    chunk.metadata["document_section"] = "부칙"
    chunk.metadata["article_full"] = f"附則 {chunk.metadata['article']}"

# 나머지는 본문
for chunk in chunks[:-5]:
    chunk.metadata["document_section"] = "본칙"
    chunk.metadata["article_full"] = chunk.metadata["article"]

In [ ]:
import json

with open("japan_chunks.jsonl", "w", encoding="utf-8") as f:
    for chunk in chunks:
        data = {
            "content": chunk.page_content,
            "metadata": chunk.metadata
        }
        f.write(json.dumps(data, ensure_ascii=False) + "\n")

print("저장 완료")

저장 완료


In [ ]:
from google.colab import files
files.download("japan_chunks.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>